# Estrategia Long Straddle sobre SPY

## 📋 Descripción de la Estrategia

El **Long Straddle** es una estrategia de opciones que consiste en:
- ✅ **Comprar** un CALL 
- ✅ **Comprar** un PUT
- ✅ Ambos con el **mismo strike** (ATM - At The Money)
- ✅ Ambos con el **mismo vencimiento**

### ¿Cuándo usar esta estrategia?

- 📈 Esperas **alta volatilidad** en el precio del subyacente
- ❓ **No sabes la dirección** del movimiento (alcista o bajista)
- 💥 Eventos importantes: earnings, decisiones de la Fed, anuncios económicos
- ⚡ Buscas **beneficio del movimiento** en cualquier dirección

### Perfil de Riesgo/Beneficio

| Aspecto | Detalle |
|---------|---------|
| **Pérdida Máxima** | Prima pagada (Call + Put) |
| **Beneficio Máximo** | Ilimitado (si el precio sube) / Sustancial (si el precio baja) |
| **Punto de Equilibrio** | Strike ± Prima Total |
| **Mejor Escenario** | Movimiento fuerte en cualquier dirección |
| **Peor Escenario** | Precio se mantiene estable cerca del strike |

---

## 🔄 Flujo de Trabajo del Notebook

**Ejecutar las celdas en este orden:**

### 1️⃣ **Conexión a TWS/IB Gateway**
- Conecta de forma robusta con reintentos automáticos
- Usa CLIENT_ID aleatorio para evitar conflictos
- Maneja desconexiones previas automáticamente

### 2️⃣ **Obtener Cadena de Opciones**
- Descarga todos los contratos disponibles para SPY
- Selecciona vencimiento automáticamente
- Retorna: `df_option_chain` y `expiry_date`

### 3️⃣ **Obtener Precio Actual de SPY**
- Sistema de fallback robusto en 3 niveles:
  1. Intenta IBKR (delayed/frozen data)
  2. Si falla → Yahoo Finance
  3. Si falla → Precio manual de seguridad
- Retorna: `spy_price`

### 4️⃣ **Obtener Precios Bid/Ask de Opciones**
- Usa datos retrasados (delayed) - **GRATUITO**
- Filtra strikes cercanos al ATM
- Procesa ~40 contratos en modo snapshot (2-4 segundos)
- Retorna: `df_with_prices`

### 5️⃣ **Desconectar**
- Cierra la conexión limpiamente
- Libera el CLIENT_ID

---

## ✅ Características Robustas Implementadas

- ✅ **Reintentos automáticos** en conexión
- ✅ **CLIENT_ID dinámico** para evitar conflictos
- ✅ **Verificación de conexión** antes de cada operación
- ✅ **Manejo de errores** detallado con mensajes claros
- ✅ **Datos retrasados** gratuitos (delayed market data)
- ✅ **Sistema de fallback** para obtener precios (IBKR → Yahoo → Manual)
- ✅ **Modo Snapshot** para obtención rápida de precios
- ✅ **Timeouts configurables**
- ✅ **Event handlers** para desconexiones
- ✅ **Mensajes de progreso** detallados

---

## 🔧 Requisitos Previos

### TWS/IB Gateway debe estar:
- ✅ **Ejecutándose**
- ✅ **Configurado para aceptar conexiones API**
  - `File → Global Configuration → API → Settings`
  - `Enable ActiveX and Socket Clients` ✓
  - Socket port: `7497` (paper) o `7496` (live)
  - Trusted IPs: `127.0.0.1`

### Librerías Python requeridas:
```python
pip install ib_insync yfinance pandas nest_asyncio
```

---

## 🚀 ¡Comienza Aquí!

**Ejecuta la siguiente celda para conectar con TWS/IB Gateway**

In [ ]:
# Instalación de ib_insync si es necesario
# !pip install ib_insync

from ib_insync import IB, util
import asyncio
import nest_asyncio
import random
import time

# Permitir event loops anidados en Jupyter
nest_asyncio.apply()

# Configuración de conexión
HOST = "127.0.0.1"
PORT = 7497          # Paper trading (7497) o Live (7496)

# Crear instancia global de IB
ib = IB()

# Configurar handlers para manejar desconexiones
def on_disconnected():
    print("⚠ Desconectado de TWS/IB Gateway")

ib.disconnectedEvent += on_disconnected

async def connect_to_ib_async(max_retries=3):
    """
    Conecta a TWS/IB Gateway de forma robusta con reintentos
    
    Parámetros:
    - max_retries: Número máximo de intentos de conexión
    """
    
    for attempt in range(max_retries):
        try:
            # Si ya está conectado, verificar que funcione
            if ib.isConnected():
                try:
                    # Test simple para verificar que la conexión funciona
                    accounts = ib.managedAccounts()
                    print(f"✓ Ya conectado - clientId: {ib.client.clientId}")
                    print(f"  - Server version: {ib.client.serverVersion()}")
                    print(f"  - Managed accounts: {accounts}")
                    return True
                except Exception:
                    # La conexión está corrupta, desconectar
                    print("⚠ Conexión existente corrupta, desconectando...")
                    try:
                        ib.disconnect()
                    except:
                        pass
                    await asyncio.sleep(2)
            
            # Generar CLIENT_ID único basado en timestamp
            # Esto evita conflictos con conexiones previas
            client_id = random.randint(100, 999)
            
            print(f"\nIntento {attempt + 1}/{max_retries}: Conectando con clientId={client_id}...")
            
            # Intentar conectar con timeout más largo
            await ib.connectAsync(HOST, PORT, clientId=client_id, timeout=20, readonly=False)
            
            # Verificar que la conexión funciona
            await asyncio.sleep(1)
            accounts = ib.managedAccounts()
            
            print("✓ Conectado exitosamente")
            print(f"  - Client ID: {client_id}")
            print(f"  - Server version: {ib.client.serverVersion()}")
            print(f"  - Managed accounts: {accounts}")
            
            return True
            
        except Exception as e:
            error_msg = str(e)
            print(f"✗ Error en intento {attempt + 1}: {error_msg}")
            
            # Limpiar conexión fallida
            try:
                if ib.isConnected():
                    ib.disconnect()
            except:
                pass
            
            # Si es el último intento, mostrar mensaje de ayuda
            if attempt == max_retries - 1:
                print("\n" + "="*80)
                print("ERROR: No se pudo conectar a TWS/IB Gateway")
                print("="*80)
                print("\nVerifica lo siguiente:")
                print("  1. TWS o IB Gateway está ejecutándose")
                print(f"  2. Está configurado para aceptar conexiones API en el puerto {PORT}")
                print("  3. La configuración API permite conexiones desde 127.0.0.1")
                print("  4. No hay otras conexiones activas bloqueando el acceso")
                print("\nPasos para configurar TWS:")
                print("  - File -> Global Configuration -> API -> Settings")
                print("  - Marcar 'Enable ActiveX and Socket Clients'")
                print(f"  - Socket port: {PORT}")
                print("  - Trusted IP addresses: 127.0.0.1")
                print("="*80)
                return False
            
            # Esperar antes del siguiente intento
            wait_time = (attempt + 1) * 2
            print(f"  Esperando {wait_time} segundos antes del siguiente intento...")
            await asyncio.sleep(wait_time)
    
    return False

# Ejecutar la conexión
print("Iniciando conexión a TWS/IB Gateway...")
print("="*80)
connection_successful = await connect_to_ib_async(max_retries=3)

if connection_successful:
    print("\n✓ Listo para operar")
else:
    print("\n✗ No se pudo establecer conexión. Revisa los mensajes anteriores.")

Iniciando conexión a TWS/IB Gateway...

Intento 1/3: Conectando con clientId=178...
✓ Conectado exitosamente
  - Client ID: 178
  - Server version: 176
  - Managed accounts: ['DUP064233']

✓ Listo para operar


Error 10089, reqId 7: Los datos de mercado solicitados requieren suscripciones adicionales para API. Consulte el enlace en 'Conexiones de Datos de Mercado' para ver m\u00e1s detalles.SPY ARCA/TOP/ALL, contract: Stock(conId=756733, symbol='SPY', exchange='SMART', primaryExchange='ARCA', currency='USD', localSymbol='SPY', tradingClass='SPY')
Error 300, reqId 7: No se ha encontrado EId con el  tickerId:7
Error 10091, reqId 50: Parte de los datos de mercado solicitados requiere suscripciones adicionales para API. Consulte el enlace en 'Conexiones a datos de mercado' para ver m\u00e1s detalles.SPY ARCA/TOP/ALL, contract: Option(conId=841979601, symbol='SPY', lastTradeDateOrContractMonth='20260107', strike=672.0, right='C', multiplier='100', exchange='SMART', currency='USD', localSymbol='SPY   260107C00672000', tradingClass='SPY')
Error 10091, reqId 51: Parte de los datos de mercado solicitados requiere suscripciones adicionales para API. Consulte el enlace en 'Conexiones a datos de mercado'

---

## 1️⃣ Conexión a TWS/IB Gateway

### 🎯 Objetivo
Establecer una conexión robusta y confiable con TWS (Trader Workstation) o IB Gateway.

### 📝 ¿Qué hace esta celda?

1. **Importa librerías necesarias**
   - `ib_insync`: Interfaz Python para Interactive Brokers
   - `nest_asyncio`: Permite event loops anidados en Jupyter
   - `asyncio`: Programación asíncrona

2. **Configura la conexión**
   - HOST: `127.0.0.1` (localhost)
   - PORT: `7497` (Paper Trading) o `7496` (Live Trading)
   - CLIENT_ID: Generado aleatoriamente para evitar conflictos

3. **Función `connect_to_ib_async()`**
   - ✅ Verifica si ya existe una conexión activa
   - ✅ Reintentos automáticos (máx 3 intentos)
   - ✅ Manejo de conexiones corruptas
   - ✅ Timeouts configurables (20 segundos)
   - ✅ Mensajes de ayuda si falla la conexión

4. **Event Handlers**
   - Detecta desconexiones automáticamente
   - Muestra advertencias cuando se pierde la conexión

### ⚙️ Configuración de MarketDataType

Por defecto, se usa **MarketDataType 4** (Frozen/Delayed):
- ✅ **GRATUITO** - No requiere suscripción a datos en tiempo real
- ✅ Datos retrasados de 15 minutos
- ✅ Ideal para Paper Trading y desarrollo
- ✅ Disponible fuera del horario de mercado

### 🔍 ¿Qué variables crea?

- `ib`: Objeto global de conexión (se usa en todas las celdas siguientes)

### ✅ Resultado esperado

```
✓ Conectado exitosamente
  - Client ID: XXX
  - Server version: XXX
  - Managed accounts: [...]
  
✓ Listo para operar
```

### ⚠️ Si falla la conexión

Verifica:
1. TWS/IB Gateway está ejecutándose
2. Puerto configurado correctamente (7497 para paper)
3. API habilitada en configuración
4. IP 127.0.0.1 en lista de IPs confiables

---

**💡 Ejecuta esta celda primero antes de continuar**

In [2]:
from datetime import datetime
import pandas as pd
from ib_insync import Stock, Contract

async def verify_connection():
    """Verifica que la conexión esté activa y funcionando"""
    if not ib.isConnected():
        raise RuntimeError("No hay conexión activa. Ejecuta primero la celda de conexión.")
    
    try:
        # Test simple para verificar que la conexión funciona
        ib.managedAccounts()
    except Exception as e:
        raise RuntimeError(f"Conexión corrupta: {e}. Ejecuta nuevamente la celda de conexión.")

async def get_option_chain_async(symbol="SPY", skip_expiries=1):
    """
    Obtiene la cadena de opciones de forma robusta
    
    Parámetros:
    - symbol: Símbolo del subyacente (default: SPY)
    - skip_expiries: Número de vencimientos a saltar desde hoy (1 = siguiente vencimiento)
    
    Returns:
    - Tuple: (df_chain, expiry) - DataFrame con la cadena y fecha de vencimiento
    """
    
    print("="*80)
    print(f"OBTENIENDO CADENA DE OPCIONES: {symbol}")
    print("="*80 + "\n")
    
    # Verificar conexión
    await verify_connection()
    print(f"✓ Conexión activa (clientId={ib.client.clientId})\n")
    
    try:
        # -----------------------------
        # 1) Definir el subyacente
        # -----------------------------
        print(f"1. Cualificando contrato del subyacente {symbol}...")
        underlying = Stock(symbol, "SMART", "USD")
        await ib.qualifyContractsAsync(underlying)
        print(f"   ✓ {underlying.symbol} - ConId: {underlying.conId}\n")
        
        # -----------------------------
        # 2) Obtener parámetros de opciones
        # -----------------------------
        print("2. Solicitando parámetros de cadenas de opciones...")
        chains = await ib.reqSecDefOptParamsAsync(symbol, "", "STK", underlying.conId)
        
        if not chains:
            raise RuntimeError(f"No se encontraron cadenas de opciones para {symbol}")
        
        # Elegir cadena estándar (no FLEX)
        chain = next((c for c in chains if c.exchange == "SMART" and c.tradingClass == symbol), None)
        
        if not chain:
            raise RuntimeError(f"No se encontró cadena estándar para {symbol}")
        
        print(f"   ✓ Cadena: exchange={chain.exchange}, tradingClass={chain.tradingClass}, multiplier={chain.multiplier}\n")
        
        # -----------------------------
        # 3) Seleccionar vencimiento
        # -----------------------------
        expirations = sorted(chain.expirations)
        today = datetime.now().strftime("%Y%m%d")
        
        # Filtrar vencimientos futuros
        future_expiries = [e for e in expirations if e > today]
        
        if not future_expiries:
            raise RuntimeError("No hay vencimientos futuros disponibles")
        
        # Seleccionar vencimiento (skip_expiries para evitar 0DTE si es necesario)
        expiry_index = min(skip_expiries, len(future_expiries) - 1)
        expiry = future_expiries[expiry_index]
        
        print(f"3. Vencimiento seleccionado: {expiry}")
        print(f"   Vencimientos disponibles: {len(future_expiries)}")
        print(f"   Primeros 5: {future_expiries[:5]}\n")
        
        # -----------------------------
        # 4) Obtener contratos de opciones
        # -----------------------------
        print("4. Solicitando contratos de opciones a IBKR...")
        print("   (esto puede tomar unos segundos...)\n")
        
        tmpl = Contract(
            secType="OPT",
            symbol=symbol,
            currency="USD",
            exchange="SMART",
            lastTradeDateOrContractMonth=expiry,
            multiplier=str(chain.multiplier),
            tradingClass=chain.tradingClass,
        )
        
        details = await ib.reqContractDetailsAsync(tmpl)
        
        if not details:
            raise RuntimeError(
                f"No se recibieron contratos para {symbol} vencimiento {expiry}. "
                "Verifica permisos de market data o intenta otro vencimiento."
            )
        
        print(f"   ✓ Recibidos {len(details)} contratos\n")
        
        # -----------------------------
        # 5) Construir DataFrame
        # -----------------------------
        print("5. Procesando contratos...")
        
        rows = []
        for d in details:
            c = d.contract
            if c.symbol != symbol or c.secType != "OPT":
                continue
            rows.append({
                "expiry": c.lastTradeDateOrContractMonth,
                "right": c.right,
                "strike": c.strike,
                "conId": c.conId,
                "localSymbol": c.localSymbol,
                "exchange": c.exchange,
                "tradingClass": getattr(c, "tradingClass", None),
            })
        
        df = pd.DataFrame(rows)
        
        # Limpiar y filtrar
        df = df[df["expiry"] == expiry]
        df = df[df["right"].isin(["C", "P"])]
        df = df.dropna(subset=["strike", "conId"])
        df["strike"] = df["strike"].astype(float)
        
        print(f"   ✓ {len(df)} contratos procesados\n")
        
        # -----------------------------
        # 6) Crear tabla Option Chain
        # -----------------------------
        print("6. Creando tabla de option chain...")
        
        df_calls = (
            df[df["right"] == "C"][["strike", "localSymbol", "conId"]]
            .rename(columns={"localSymbol": "call_localSymbol", "conId": "call_conId"})
        )
        df_puts = (
            df[df["right"] == "P"][["strike", "localSymbol", "conId"]]
            .rename(columns={"localSymbol": "put_localSymbol", "conId": "put_conId"})
        )
        
        df_chain = (
            df_calls.merge(df_puts, on="strike", how="outer")
            .sort_values("strike")
            .reset_index(drop=True)
        )
        
        # Mostrar resumen
        print(f"\n{'='*80}")
        print(f"OPTION CHAIN {symbol} - Expiry: {expiry}")
        print(f"{'='*80}")
        print(f"Total strikes: {len(df_chain)}")
        print(f"Rango: ${df_chain['strike'].min():.0f} - ${df_chain['strike'].max():.0f}")
        print(f"\nPrimeros 10 strikes:")
        print(df_chain.head(10)[['strike', 'call_localSymbol', 'put_localSymbol']].to_string(index=False))
        print(f"{'='*80}\n")
        
        print("✓ Cadena de opciones obtenida exitosamente\n")
        
        return df_chain, expiry
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}\n")
        raise

# Ejecutar la función
try:
    df_option_chain, expiry_date = await get_option_chain_async("SPY", skip_expiries=2)
    print(f"Variable 'df_option_chain' creada con {len(df_option_chain)} strikes")
    print(f"Variable 'expiry_date' = {expiry_date}")
except Exception as e:
    print(f"No se pudo obtener la cadena de opciones: {e}")

OBTENIENDO CADENA DE OPCIONES: SPY

✓ Conexión activa (clientId=178)

1. Cualificando contrato del subyacente SPY...
   ✓ SPY - ConId: 756733

2. Solicitando parámetros de cadenas de opciones...
   ✓ Cadena: exchange=SMART, tradingClass=SPY, multiplier=100

3. Vencimiento seleccionado: 20260107
   Vencimientos disponibles: 32
   Primeros 5: ['20260105', '20260106', '20260107', '20260108', '20260109']

4. Solicitando contratos de opciones a IBKR...
   (esto puede tomar unos segundos...)

   ✓ Recibidos 274 contratos

5. Procesando contratos...
   ✓ 274 contratos procesados

6. Creando tabla de option chain...

OPTION CHAIN SPY - Expiry: 20260107
Total strikes: 137
Rango: $500 - $900

Primeros 10 strikes:
 strike      call_localSymbol       put_localSymbol
  500.0 SPY   260107C00500000 SPY   260107P00500000
  505.0 SPY   260107C00505000 SPY   260107P00505000
  510.0 SPY   260107C00510000 SPY   260107P00510000
  515.0 SPY   260107C00515000 SPY   260107P00515000
  520.0 SPY   260107C005200

---

## 2️⃣ Obtener Cadena de Opciones

### 🎯 Objetivo
Descargar la cadena completa de opciones para SPY con un vencimiento específico.

### 📝 ¿Qué hace esta celda?

1. **Función `verify_connection()`**
   - Verifica que la conexión con IBKR esté activa
   - Lanza error si la conexión está corrupta o inactiva

2. **Función `get_option_chain_async()`**
   - **Parámetros:**
     - `symbol`: Símbolo del subyacente (default: "SPY")
     - `skip_expiries`: Número de vencimientos a saltar (1 = siguiente vencimiento, 2 = el siguiente, etc.)
   
   - **Proceso:**
     1. Cualifica el contrato del subyacente (SPY)
     2. Solicita parámetros de opciones disponibles
     3. Selecciona la cadena estándar (exchange=SMART)
     4. Filtra vencimientos futuros
     5. Selecciona vencimiento automáticamente
     6. Descarga todos los contratos (CALLs y PUTs)
     7. Construye DataFrame con la cadena completa

3. **Estructura de datos retornada**
   - Tabla pivoteada: cada fila = un strike
   - Columnas:
     - `strike`: Precio de ejercicio
     - `call_localSymbol`: Símbolo local del CALL
     - `call_conId`: ID del contrato CALL
     - `put_localSymbol`: Símbolo local del PUT
     - `put_conId`: ID del contrato PUT

### 🔍 ¿Qué variables crea?

- `df_option_chain`: DataFrame con la cadena completa de opciones
- `expiry_date`: Fecha de vencimiento seleccionada (formato: YYYYMMDD)

### ⚙️ Parámetros configurables

```python
# Ejemplo: Obtener el tercer vencimiento (saltar los 2 primeros)
df_option_chain, expiry_date = await get_option_chain_async("SPY", skip_expiries=2)
```

### ✅ Resultado esperado

```
================================================================================
OPTION CHAIN SPY - Expiry: YYYYMMDD
================================================================================
Total strikes: 254
Rango: $345 - $890

Primeros 10 strikes:
 strike      call_localSymbol       put_localSymbol
  345.0 SPY   251231C00345000 SPY   251231P00345000
  ...
================================================================================

✓ Cadena de opciones obtenida exitosamente

Variable 'df_option_chain' creada con 254 strikes
Variable 'expiry_date' = 20251231
```

### ⏱️ Tiempo de ejecución

- Típicamente: **5-10 segundos**
- Depende de la cantidad de strikes disponibles

### 💡 Notas importantes

- ✅ Filtra automáticamente vencimientos pasados
- ✅ Evita 0DTE usando `skip_expiries=1` o mayor
- ✅ Obtiene contratos estándar (no FLEX)
- ✅ Maneja errores de permisos de market data

---

**💡 Ejecuta esta celda después de conectar con IBKR**

In [3]:
import asyncio
from ib_insync import Stock

async def get_spy_price_async():
    """
    Obtiene el precio actual de SPY con fallback robusto:
    1. Intenta obtenerlo de IBKR (datos delayed/frozen)
    2. Si falla, descarga de Yahoo Finance
    3. Si falla, usa precio manual de seguridad
    
    Returns:
    - float: Precio actual de SPY
    """
    
    print("="*80)
    print("OBTENIENDO PRECIO DE SPY (Con Fallback Robusto)")
    print("="*80 + "\n")
    
    # =========================================================================
    # MÉTODO 1: Obtener de IBKR
    # =========================================================================
    print("📊 MÉTODO 1: Intentando obtener precio desde IBKR...")
    print("-" * 80)
    
    if ib.isConnected():
        try:
            # Configurar datos retrasados/congelados (MarketDataType 4)
            ib.reqMarketDataType(4)
            print("✓ Conexión activa - MarketDataType(4) configurado\n")
            
            # Crear contrato de SPY
            print("1. Cualificando contrato SPY...")
            spy = Stock("SPY", "SMART", "USD")
            await ib.qualifyContractsAsync(spy)
            print(f"   ✓ Contrato cualificado - ConId: {spy.conId}\n")
            
            # Solicitar datos de mercado (snapshot)
            print("2. Solicitando precio de mercado...")
            spy_ticker = ib.reqMktData(spy, "", snapshot=True, regulatorySnapshot=False)
            
            # Esperar a recibir datos (máximo 3 segundos)
            spy_price = None
            for i in range(15):
                await asyncio.sleep(0.2)
                
                # Intentar obtener precio en orden de prioridad
                if spy_ticker.last and spy_ticker.last > 0:
                    spy_price = spy_ticker.last
                    price_type = "Last"
                    break
                elif spy_ticker.close and spy_ticker.close > 0:
                    spy_price = spy_ticker.close
                    price_type = "Close"
                    break
                elif spy_ticker.bid and spy_ticker.bid > 0:
                    spy_price = spy_ticker.bid
                    price_type = "Bid"
                    break
            
            # Cancelar suscripción
            ib.cancelMktData(spy)
            
            # Verificar resultado
            if spy_price and spy_price > 0:
                print(f"✅ ÉXITO - Precio obtenido de IBKR")
                print("="*80)
                print(f"PRECIO SPY: ${spy_price:.2f}")
                print(f"Fuente: IBKR ({price_type})")
                print("="*80 + "\n")
                return spy_price
            else:
                print("⚠ IBKR no devolvió precio válido\n")
                
        except Exception as e:
            print(f"⚠ Error al obtener precio de IBKR: {e}\n")
    else:
        print("⚠ No hay conexión activa con IBKR\n")
    
    # =========================================================================
    # MÉTODO 2: Obtener de Yahoo Finance
    # =========================================================================
    print("📊 MÉTODO 2: Intentando obtener precio desde Yahoo Finance...")
    print("-" * 80)
    
    try:
        # Intentar importar yfinance
        try:
            import yfinance as yf
        except ImportError:
            print("⚠ Librería 'yfinance' no instalada.")
            print("  Instalando automáticamente...")
            import subprocess
            import sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
            import yfinance as yf
            print("✓ yfinance instalado correctamente\n")
        
        print("1. Descargando datos de Yahoo Finance...")
        spy_ticker = yf.Ticker("SPY")
        
        # Obtener datos de diferentes fuentes
        spy_info = spy_ticker.info
        spy_history = spy_ticker.history(period="1d")
        
        spy_price = None
        price_type = None
        
        # Prioridad: regularMarketPrice > currentPrice > previousClose > Close del history
        if 'regularMarketPrice' in spy_info and spy_info['regularMarketPrice']:
            spy_price = spy_info['regularMarketPrice']
            price_type = "Regular Market Price"
        elif 'currentPrice' in spy_info and spy_info['currentPrice']:
            spy_price = spy_info['currentPrice']
            price_type = "Current Price"
        elif 'previousClose' in spy_info and spy_info['previousClose']:
            spy_price = spy_info['previousClose']
            price_type = "Previous Close"
        elif not spy_history.empty:
            spy_price = spy_history['Close'].iloc[-1]
            price_type = "Last Close (History)"
        
        if spy_price and spy_price > 0:
            print(f"✅ ÉXITO - Precio obtenido de Yahoo Finance")
            print("="*80)
            print(f"PRECIO SPY: ${spy_price:.2f}")
            print(f"Fuente: Yahoo Finance ({price_type})")
            print("="*80 + "\n")
            return float(spy_price)
        else:
            print("⚠ Yahoo Finance no devolvió precio válido\n")
            
    except Exception as e:
        print(f"⚠ Error al obtener precio de Yahoo Finance: {e}\n")
    
    # =========================================================================
    # MÉTODO 3: Precio Manual de Seguridad
    # =========================================================================
    print("📊 MÉTODO 3: Usando precio manual de seguridad...")
    print("-" * 80)
    
    # Precio de seguridad (actualizar manualmente si es necesario)
    # Este es un precio aproximado del SPY a finales de diciembre 2024
    safety_price = 595.00
    
    print(f"⚠️ ADVERTENCIA: Usando precio manual de seguridad")
    print("="*80)
    print(f"PRECIO SPY: ${safety_price:.2f}")
    print(f"Fuente: Precio Manual de Seguridad")
    print("="*80)
    print("\n⚠️  IMPORTANTE: Este es un precio de respaldo.")
    print("   Verifica la conexión IBKR o la disponibilidad de Yahoo Finance")
    print("   para obtener precios actualizados en tiempo real.\n")
    
    return safety_price

# Ejecutar la función
spy_price = await get_spy_price_async()

print(f"✓ Variable 'spy_price' = ${spy_price:.2f}")

OBTENIENDO PRECIO DE SPY (Con Fallback Robusto)

📊 MÉTODO 1: Intentando obtener precio desde IBKR...
--------------------------------------------------------------------------------
✓ Conexión activa - MarketDataType(4) configurado

1. Cualificando contrato SPY...
   ✓ Contrato cualificado - ConId: 756733

2. Solicitando precio de mercado...
⚠ IBKR no devolvió precio válido

📊 MÉTODO 2: Intentando obtener precio desde Yahoo Finance...
--------------------------------------------------------------------------------
1. Descargando datos de Yahoo Finance...
✅ ÉXITO - Precio obtenido de Yahoo Finance
PRECIO SPY: $681.92
Fuente: Yahoo Finance (Regular Market Price)

✓ Variable 'spy_price' = $681.92


---

## 3️⃣ Obtener Precio Actual de SPY

### 🎯 Objetivo
Obtener el precio actual del subyacente (SPY) para identificar el strike ATM (At The Money).

### 📝 ¿Qué hace esta celda?

Esta celda implementa un **sistema de fallback robusto en 3 niveles** para asegurar que siempre obtengamos un precio de referencia:

#### **MÉTODO 1: IBKR (Prioridad Alta)**
- Usa la conexión activa con IBKR
- Solicita datos de mercado retrasados (MarketDataType 4)
- Prioridad de precios:
  1. `Last` (último precio negociado)
  2. `Close` (precio de cierre)
  3. `Bid` (mejor oferta de compra)
- ✅ **Gratis** - No requiere suscripción a datos en tiempo real

#### **MÉTODO 2: Yahoo Finance (Fallback)**
- Si IBKR no devuelve precio válido
- Descarga datos usando la librería `yfinance`
- Instala automáticamente `yfinance` si no está disponible
- Prioridad de precios:
  1. `regularMarketPrice` (precio de mercado regular)
  2. `currentPrice` (precio actual)
  3. `previousClose` (cierre anterior)
  4. `Close` del historial
- ✅ **Gratis** - No requiere API key

#### **MÉTODO 3: Precio Manual (Última Opción)**
- Si ambos métodos anteriores fallan
- Usa precio de seguridad hardcodeado (~$595.00)
- ⚠️ **Advertencia clara al usuario** para verificar conexión
- Permite que el notebook continúe funcionando

### 🔍 ¿Qué variables crea?

- `spy_price`: Precio actual de SPY (float)

### ✅ Resultado esperado

**Caso exitoso (IBKR):**
```
✅ ÉXITO - Precio obtenido de IBKR
================================================================================
PRECIO SPY: $689.51
Fuente: IBKR (Last)
================================================================================
```

**Caso exitoso (Yahoo):**
```
✅ ÉXITO - Precio obtenido de Yahoo Finance
================================================================================
PRECIO SPY: $689.51
Fuente: Yahoo Finance (Regular Market Price)
================================================================================
```

**Caso fallback:**
```
⚠️ ADVERTENCIA: Usando precio manual de seguridad
================================================================================
PRECIO SPY: $595.00
Fuente: Precio Manual de Seguridad
================================================================================

⚠️  IMPORTANTE: Este es un precio de respaldo.
   Verifica la conexión IBKR o la disponibilidad de Yahoo Finance
```

### ⏱️ Tiempo de ejecución

- IBKR: **1-3 segundos**
- Yahoo Finance: **2-5 segundos**
- Manual: **Instantáneo**

### 💡 Notas importantes

- ✅ **Triple capa de seguridad** - Siempre obtendrás un precio
- ✅ **No bloquea el flujo** - El notebook puede continuar
- ✅ **Instalación automática** de dependencias (yfinance)
- ✅ **Mensajes claros** indicando la fuente del precio
- ⚠️ Si ves el precio manual, verifica tu conexión

### 🔧 ¿Cuándo actualizar el precio manual?

Si el precio de SPY cambia significativamente (~>10%), actualiza esta línea en el código:

```python
safety_price = 595.00  # <-- Actualizar aquí
```

---

**💡 Ejecuta esta celda después de obtener la cadena de opciones**

In [4]:
import asyncio
import pandas as pd
from ib_insync import Option, Stock, util

async def get_option_prices_async(df_chain, expiry_date, center_strike=None, max_strikes=20, reference_price=None):
    """
    Obtiene precios bid/ask de forma robusta usando Snapshot y MarketDataType 4 (Frozen).
    
    Parámetros:
    - df_chain: DataFrame con la cadena de opciones
    - expiry_date: Fecha de vencimiento
    - center_strike: Strike central específico (opcional)
    - max_strikes: Número máximo de strikes a procesar
    - reference_price: Precio de referencia del subyacente (si ya se obtuvo previamente)
    """
    
    print("="*80)
    print(f"OBTENIENDO PRECIOS DE OPCIONES - Expiry: {expiry_date}")
    print("="*80 + "\n")
    
    # Configurar datos congelados/retrasados (Frozen es mejor para pruebas off-hours o delayed)
    ib.reqMarketDataType(4) 
    print("✓ Configurado: MarketDataType(4) - Frozen/Delayed (Más robusto para Paper Trading)\n")
    
    try:
        # -----------------------------
        # 1) Obtener precio de referencia (SPY)
        # -----------------------------
        spy_price = reference_price  # Usar el precio proporcionado como parámetro
        
        if spy_price:
            print(f"1. Usando precio de referencia proporcionado: ${spy_price:.2f}")
            print("   (Obtenido previamente de Yahoo Finance o IBKR)\n")
        else:
            print("1. Obteniendo precio de referencia del subyacente (SPY)...")
            
            try:
                spy = Stock("SPY", "SMART", "USD")
                await ib.qualifyContractsAsync(spy)
                
                # Usamos snapshot=True para asegurar un dato puntual
                spy_ticker = ib.reqMktData(spy, "", True, False)
                
                # Esperar hasta obtener un dato válido (máx 2 seg)
                for _ in range(10):
                    await asyncio.sleep(0.2)
                    # Prioridad: Last -> Close -> MarketPrice
                    if spy_ticker.last and spy_ticker.last > 0:
                        spy_price = spy_ticker.last
                        break
                    elif spy_ticker.close and spy_ticker.close > 0:
                        spy_price = spy_ticker.close
                        break
                
                # Si sigue fallando, intentar marketPrice() genérico de ib_insync
                if not spy_price:
                    spy_price = spy_ticker.marketPrice()
                    
                if spy_price and spy_price > 0:
                    print(f"   ✓ Precio SPY detectado: ${spy_price:.2f}\n")
                else:
                    print("   ⚠ ADVERTENCIA: No se pudo obtener precio SPY. Usando strike central por defecto.\n")
                    
            except Exception as e:
                print(f"   ⚠ Error obteniendo SPY: {e}\n")
        
        # -----------------------------
        # 2) Seleccionar strikes
        # -----------------------------
        print("2. Seleccionando strikes...")
        
        # Determinar strike central
        if center_strike is None:
            if spy_price and spy_price > 0:
                temp_df = df_chain.copy()
                temp_df['distance'] = abs(temp_df['strike'] - spy_price)
                center_strike = temp_df.loc[temp_df['distance'].idxmin(), 'strike']
            else:
                # Fallback si no hay precio de SPY: usar la mediana de la cadena
                center_strike = df_chain['strike'].median()
                print(f"   (Usando mediana de la cadena como referencia: {center_strike})")
        
        # Filtrar strikes alrededor del centro
        strike_range = max_strikes // 2
        df_filtered = df_chain.copy().sort_values('strike').reset_index(drop=True)
        
        df_filtered['distance'] = abs(df_filtered['strike'] - center_strike)
        center_idx = df_filtered['distance'].idxmin()
        
        start_idx = max(0, center_idx - strike_range)
        end_idx = min(len(df_filtered), center_idx + strike_range + 1)
        df_filtered = df_filtered.iloc[start_idx:end_idx].copy()
        df_filtered = df_filtered.drop('distance', axis=1).reset_index(drop=True)
        
        print(f"   ✓ Strike central: ${center_strike:.2f}")
        print(f"   ✓ Rango: ${df_filtered['strike'].min():.0f} - ${df_filtered['strike'].max():.0f}")
        print(f"   ✓ {len(df_filtered)} strikes seleccionados\n")
        
        # -----------------------------
        # 3) Crear y Cualificar contratos
        # -----------------------------
        print("3. Creando y cualificando contratos...")
        
        contracts = []
        contract_info = []  # Lista en lugar de diccionario (evita problemas de hash)
        
        for idx, row in df_filtered.iterrows():
            strike = row['strike']
            
            # CALL
            if pd.notna(row.get('call_conId')):
                call = Option('SPY', expiry_date, strike, 'C', 'SMART', tradingClass='SPY')
                contracts.append(call)
                contract_info.append(('call', idx))  # Guardar tipo y índice
            
            # PUT
            if pd.notna(row.get('put_conId')):
                put = Option('SPY', expiry_date, strike, 'P', 'SMART', tradingClass='SPY')
                contracts.append(put)
                contract_info.append(('put', idx))  # Guardar tipo y índice
        
        qualified = await ib.qualifyContractsAsync(*contracts)
        print(f"   ✓ {len(qualified)} contratos listos para solicitar datos\n")
        
        # -----------------------------
        # 4) Obtener precios (SNAPSHOT / PARALELO)
        # -----------------------------
        print("4. Solicitando Snapshots de mercado (Paralelo)...")
        
        # Inicializar columnas en el DF
        cols_to_init = ['call_bid', 'call_ask', 'call_last', 'call_volume', 
                       'put_bid', 'put_ask', 'put_last', 'put_volume']
        for col in cols_to_init:
            df_filtered[col] = None

        # SOLICITUD MASIVA (Mucho más rápido)
        # Usamos snapshot=True. Esto es clave para cuentas sin datos live.
        tickers_list = []
        for contract in qualified:
            # snapshot=True, regulatorySnapshot=False
            t = ib.reqMktData(contract, "", True, False)
            tickers_list.append(t)
        
        print(f"   ⏳ Esperando datos (2-4 segundos)...")
        # Esperamos un tiempo prudencial para que lleguen los snapshots
        await asyncio.sleep(20) 
        
        # Opcional: Espera activa inteligente (esperar a que la mayoría tenga datos)
        for _ in range(20):  # Máx 4 segundos extra
            filled = sum(1 for t in tickers_list if t.last or t.bid or t.ask or t.close)
            if filled >= len(qualified) * 0.9:  # Si tenemos el 90% de los datos
                break
            await asyncio.sleep(0.2)

        # -----------------------------
        # 5) Procesar resultados
        # -----------------------------
        valid_count = 0

        # Procesamos los resultados usando índices
        # contract_info, qualified y tickers_list están alineados por índice
        for i, (qualified_contract, ticker) in enumerate(zip(qualified, tickers_list)):

            # Obtener información usando el índice
            if i >= len(contract_info):
                continue
                
            opt_type, df_idx = contract_info[i]

            # Lógica de precios (Priorizamos Last, luego Close para Delayed)
            bid = ticker.bid if (ticker.bid and ticker.bid > 0) else None
            ask = ticker.ask if (ticker.ask and ticker.ask > 0) else None
            last = ticker.last if (ticker.last and ticker.last > 0) else ticker.close
            volume = ticker.volume

            # Asignar al DataFrame
            prefix = 'call' if opt_type == 'call' else 'put'

            df_filtered.at[df_idx, f'{prefix}_bid'] = bid
            df_filtered.at[df_idx, f'{prefix}_ask'] = ask
            df_filtered.at[df_idx, f'{prefix}_last'] = last
            df_filtered.at[df_idx, f'{prefix}_volume'] = volume

            if bid or ask or last:
                valid_count += 1

            # Limpiar suscripción (buena práctica)
            ib.cancelMktData(qualified_contract)

        # -----------------------------
        # 6) Mostrar resultados
        # -----------------------------
        print(f"\n{'='*100}")
        print(f"PRECIOS DE OPCIONES SPY - Expiry: {expiry_date} (Snapshot Mode)")
        print(f"{'='*100}\n")
        
        display_cols = ['strike', 'call_bid', 'call_ask', 'call_last', 'put_bid', 'put_ask', 'put_last']
        
        # Formatear para imprimir bonito (rellenar NaN con guiones)
        df_print = df_filtered[display_cols].fillna('-')
        print(df_print.to_string(index=False))
        
        print(f"\n{'='*100}")
        print(f"Strikes: {len(df_filtered)} | Contratos con precios: {valid_count}/{len(qualified)}")
        print(f"{'='*100}\n")
        
        return df_filtered
        
    except Exception as e:
        print(f"\n✗ ERROR CRÍTICO: {e}")
        import traceback
        traceback.print_exc()
        return df_chain  # Devolver original en caso de error

# ---------------------------------------------------------
# Bloque de ejecución principal
# ---------------------------------------------------------
# Ejecutar la función pasando el precio obtenido de Yahoo Finance
df_with_prices = await get_option_prices_async(df_option_chain, expiry_date, reference_price=spy_price)

OBTENIENDO PRECIOS DE OPCIONES - Expiry: 20260107

✓ Configurado: MarketDataType(4) - Frozen/Delayed (Más robusto para Paper Trading)

1. Usando precio de referencia proporcionado: $681.92
   (Obtenido previamente de Yahoo Finance o IBKR)

2. Seleccionando strikes...
   ✓ Strike central: $682.00
   ✓ Rango: $672 - $692
   ✓ 21 strikes seleccionados

3. Creando y cualificando contratos...
   ✓ 42 contratos listos para solicitar datos

4. Solicitando Snapshots de mercado (Paralelo)...
   ⏳ Esperando datos (2-4 segundos)...

PRECIOS DE OPCIONES SPY - Expiry: 20260107 (Snapshot Mode)

 strike  call_bid  call_ask  call_last put_bid put_ask put_last
  672.0     11.63     11.92      11.32    0.77    0.79     0.88
  673.0     10.69     11.04      10.46    0.89    0.91     1.02
  674.0      9.99     10.14       9.61    1.03    1.05     1.17
  675.0      9.15      9.30       8.78    1.19    1.21     1.35
  676.0      8.22      8.50       7.98    1.37    1.39     1.54
  677.0      7.54      7.69 

C:\Users\diego\AppData\Local\Temp\ipykernel_18620\4122613910.py:200: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_print = df_filtered[display_cols].fillna('-')


---

## 4️⃣ Obtener Precios Bid/Ask de Opciones

### 🎯 Objetivo
Obtener precios de mercado (bid/ask) para las opciones cercanas al precio ATM de SPY.

### 📝 ¿Qué hace esta celda?

#### **Función `get_option_prices_async()`**

**Parámetros:**
- `df_chain`: DataFrame con la cadena de opciones (del paso anterior)
- `expiry_date`: Fecha de vencimiento (del paso anterior)
- `center_strike`: Strike central específico (opcional)
- `max_strikes`: Número máximo de strikes a procesar (default: 20)
- `reference_price`: Precio de referencia de SPY (del paso anterior)

**Proceso:**

1. **Configuración de datos**
   - Usa MarketDataType(4) - Frozen/Delayed
   - ✅ **GRATUITO** - No requiere suscripción a datos en tiempo real
   - Ideal para Paper Trading y desarrollo

2. **Selección de strikes**
   - Usa el precio de referencia de SPY
   - Encuentra el strike más cercano (ATM)
   - Selecciona `max_strikes` alrededor del ATM
   - Ejemplo: Si ATM=$690 y max_strikes=20 → strikes de $680 a $700

3. **Creación de contratos**
   - Crea objetos `Option` para cada CALL y PUT
   - Cualifica contratos con IBKR
   - Prepara para solicitud masiva de datos

4. **Solicitud de precios (MODO SNAPSHOT)**
   - ⚡ **Solicitud paralela masiva** - Todos los contratos a la vez
   - `snapshot=True` - Solicita dato puntual (no streaming)
   - Espera inteligente: 2-4 segundos con verificación del 90%
   - ✅ **Mucho más rápido** que solicitudes secuenciales

5. **Procesamiento de resultados**
   - Extrae: bid, ask, last, volume
   - Prioriza: Last → Close (para datos delayed)
   - Guarda en DataFrame con columnas separadas para CALL y PUT
   - Cancela suscripciones para liberar recursos

6. **Visualización**
   - Tabla formateada con todos los strikes
   - Precios bid/ask/last para CALLs y PUTs
   - Estadísticas de cobertura

### 🔍 ¿Qué variables crea?

- `df_with_prices`: DataFrame con strikes y precios completos

**Columnas del DataFrame:**
- `strike`: Precio de ejercicio
- `call_localSymbol`: Símbolo local del CALL
- `call_conId`: ID del contrato CALL
- `call_bid`: Precio bid del CALL
- `call_ask`: Precio ask del CALL
- `call_last`: Último precio del CALL
- `call_volume`: Volumen del CALL
- `put_localSymbol`: Símbolo local del PUT
- `put_conId`: ID del contrato PUT
- `put_bid`: Precio bid del PUT
- `put_ask`: Precio ask del PUT
- `put_last`: Último precio del PUT
- `put_volume`: Volumen del PUT

### ✅ Resultado esperado

```
====================================================================================================
PRECIOS DE OPCIONES SPY - Expiry: 20251231 (Snapshot Mode)
====================================================================================================

 strike  call_bid  call_ask  call_last  put_bid  put_ask  put_last
  680.0     10.20     11.23      11.02     0.51     0.52      0.52
  681.0      9.60     10.00       9.80     0.58     0.59      0.57
  ...
  690.0      2.59      2.60       2.59     2.28     2.30      2.28  ← ATM
  ...
  700.0      0.09      0.10       0.09     8.84    10.20      8.56

====================================================================================================
Strikes: 21 | Contratos con precios: 42/42
====================================================================================================
```

### ⏱️ Tiempo de ejecución

- **2-4 segundos** para ~40 contratos (20 strikes × 2 opciones)
- ⚡ **Mucho más rápido** que el método secuencial (~30-40 segundos)

### 💡 Ventajas del Modo Snapshot

- ✅ **Solicitud paralela masiva** - Todos los contratos simultáneamente
- ✅ **No requiere streaming** - Solo un dato puntual
- ✅ **Gratis con datos delayed** - No requiere suscripción
- ✅ **Espera inteligente** - Verifica que llegue el 90% de datos
- ✅ **Limpieza automática** - Cancela suscripciones al terminar

### ⚙️ Parámetros configurables

```python
# Obtener más strikes (ejemplo: 40 strikes)
df_with_prices = await get_option_prices_async(
    df_option_chain, 
    expiry_date, 
    max_strikes=40,
    reference_price=spy_price
)

# Forzar un strike central específico
df_with_prices = await get_option_prices_async(
    df_option_chain, 
    expiry_date, 
    center_strike=690.0,
    reference_price=spy_price
)
```

### 🎯 Uso para Long Straddle

Con estos datos puedes:
1. Identificar el strike ATM exacto
2. Ver el costo del CALL ATM (call_ask)
3. Ver el costo del PUT ATM (put_ask)
4. **Costo total = call_ask + put_ask**
5. Calcular puntos de equilibrio (strike ± costo total)

### 💡 Notas importantes

- ✅ Usa precios **delayed/frozen** - GRATUITOS
- ✅ Modo snapshot - Mucho más rápido y eficiente
- ✅ Filtra strikes cercanos al ATM automáticamente
- ⚠️ Durante horario de mercado: datos con 15 min de retraso
- ⚠️ Fuera de horario: datos "frozen" del último cierre

---

**💡 Ejecuta esta celda después de obtener el precio de SPY**

In [5]:
async def disconnect_from_ib_async():
    """Desconecta de TWS/IB Gateway de forma segura"""
    
    print("="*80)
    print("CERRANDO CONEXIÓN")
    print("="*80 + "\n")
    
    try:
        if ib.isConnected():
            client_id = ib.client.clientId
            print(f"Desconectando clientId={client_id}...")
            
            # Cancelar todas las suscripciones de market data activas
            try:
                ib.cancelMktData()
            except:
                pass
            
            # Desconectar
            ib.disconnect()
            await asyncio.sleep(1)
            
            print("✓ Desconectado exitosamente de TWS/IB Gateway")
            print(f"  ClientId {client_id} liberado\n")
        else:
            print("⚠ No hay conexión activa para cerrar\n")
            
    except Exception as e:
        print(f"✗ Error al desconectar: {e}")
        print("  (La conexión puede haberse cerrado ya)\n")

# Ejecutar desconexión
await disconnect_from_ib_async()
print("✓ Sesión finalizada")

CERRANDO CONEXIÓN

Desconectando clientId=178...
⚠ Desconectado de TWS/IB Gateway
✓ Desconectado exitosamente de TWS/IB Gateway
  ClientId 178 liberado

✓ Sesión finalizada


---

## 5️⃣ Desconectar de TWS/IB Gateway

### 🎯 Objetivo
Cerrar la conexión con IBKR de forma limpia y liberar el CLIENT_ID.

### 📝 ¿Qué hace esta celda?

#### **Función `disconnect_from_ib_async()`**

1. **Verifica si hay conexión activa**
   - Comprueba `ib.isConnected()`
   - Si no hay conexión, muestra advertencia y termina

2. **Cancela suscripciones activas**
   - Cancela todas las suscripciones de market data
   - Limpia recursos antes de desconectar
   - Previene warnings de IBKR

3. **Desconecta limpiamente**
   - Ejecuta `ib.disconnect()`
   - Espera 1 segundo para asegurar desconexión completa
   - Libera el CLIENT_ID para futuros usos

4. **Confirma desconexión**
   - Muestra CLIENT_ID liberado
   - Confirma que la sesión finalizó correctamente

### ✅ Resultado esperado

```
================================================================================
CERRANDO CONEXIÓN
================================================================================

Desconectando clientId=831...
⚠ Desconectado de TWS/IB Gateway
✓ Desconectado exitosamente de TWS/IB Gateway
  ClientId 831 liberado

✓ Sesión finalizada
```

### ⏱️ Tiempo de ejecución

- **1-2 segundos**

### 💡 ¿Por qué es importante desconectar?

1. **Libera el CLIENT_ID**
   - IBKR tiene límite de conexiones simultáneas
   - Liberar el ID permite reconectar más rápido
   - Evita conflictos de IDs en futuras sesiones

2. **Limpia recursos**
   - Cancela suscripciones de market data
   - Libera memoria y conexiones de red
   - Previene warnings y errores

3. **Buena práctica**
   - Cierre ordenado del sistema
   - Evita conexiones "zombies"
   - Facilita debugging

### 🔄 ¿Cuándo ejecutar esta celda?

- ✅ **Al terminar tu trabajo** con opciones
- ✅ **Antes de cerrar el notebook** (si vas a seguir trabajando en él)
- ✅ **Antes de reiniciar la conexión** con nuevos parámetros
- ⚠️ **NO ejecutar** si planeas seguir consultando datos inmediatamente

### 💡 Reconectar después de desconectar

Si desconectas y luego quieres volver a trabajar:

1. Vuelve a ejecutar la celda **1️⃣ Conexión a TWS/IB Gateway**
2. Eso creará una nueva conexión con un nuevo CLIENT_ID
3. Continúa normalmente con las siguientes celdas

### 📌 Notas importantes

- ✅ Puedes ejecutar esta celda múltiples veces sin problemas
- ✅ Si ya estás desconectado, solo mostrará advertencia
- ✅ No afecta los DataFrames ya creados (`df_option_chain`, `df_with_prices`, etc.)
- ⚠️ Después de desconectar, necesitas reconectar para obtener nuevos datos

---

**💡 Ejecuta esta celda cuando termines de trabajar con opciones**